<div align='center'>

# 📈 AquaVolt-AI — LSTM Water Deficit Forecasting
### Deep Learning Time Series Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umertanveer25/aquavolt-ai-pk/blob/main/notebooks/lstm_forecasting.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-aquavolt--ai--pk-black?logo=github)](https://github.com/umertanveer25/aquavolt-ai-pk)

**Umer Tanveer** · PhD Candidate, AWKUM Pakistan

</div>

---

This notebook trains an **LSTM (Long Short-Term Memory)** neural network on live AquaVolt-AI telemetry data to **forecast crop water deficit 1 hour ahead**.

The model uses 12 hours of historical satellite + weather data as input and predicts the next hour's water deficit.

> **The model automatically improves as more data accumulates.** Run this notebook again in a week for better results.

In [ ]:
!pip install -q tensorflow matplotlib seaborn pandas numpy scikit-learn

## 📡 Step 1: Load Live Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

SHEET_ID = '1c2a-3t8fF2g_PX_0ape4ASTsbr5uX0Zb6YPzT8jtuN8'
url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=Sheet1'

print('🛰️  Loading live AquaVolt-AI telemetry...')
df = pd.read_csv(url, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp')

num_cols = ['ndvi','ndwi','kc','ks','etc','water_need','air_temp','humidity','solar_rad','soil_moisture']
for c in num_cols:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')

df['hour'] = df['timestamp'].dt.floor('h')
n_hours = df['hour'].nunique()
print(f'✅ Loaded {len(df):,} records | {n_hours} unique hours')

MIN_HOURS = 48
if n_hours < MIN_HOURS:
    print(f'\n⚠️ Need at least {MIN_HOURS} hours of data for LSTM training.')
    print(f'   Current data: {n_hours} hours.')
    print(f'   Re-run this notebook after more data accumulates (approx {MIN_HOURS - n_hours} more hours needed).')
    READY = False
else:
    READY = True
    print(f'✅ Sufficient data for LSTM training!')

## ⚙️ Step 2: Feature Engineering

In [ ]:
if READY:
    features = [c for c in ['air_temp','humidity','solar_rad','ndvi','ndwi','kc','ks'] if c in df.columns]
    target = 'water_need'
    
    # Aggregate to hourly means across all sectors
    hourly = df.groupby('hour')[features + [target]].mean().dropna().reset_index()
    hourly = hourly.sort_values('hour').reset_index(drop=True)
    
    print(f'📊 Hourly aggregated dataset: {len(hourly)} rows')
    print(f'📋 Features: {features}')
    print(f'🎯 Target: {target}')
    display(hourly.tail(5))

## 🧠 Step 3: Build and Train LSTM Model

In [ ]:
if READY:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    
    LOOKBACK = min(12, len(hourly) // 4)  # Adapt lookback to data size
    
    # Normalize
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_data = scaler_X.fit_transform(hourly[features].values)
    y_data = scaler_y.fit_transform(hourly[[target]].values)
    
    # Create sequences
    def create_sequences(X, y, lookback):
        Xs, ys = [], []
        for i in range(lookback, len(X)):
            Xs.append(X[i-lookback:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)
    
    X_seq, y_seq = create_sequences(X_data, y_data, LOOKBACK)
    
    # 80/20 chronological split
    split = int(0.8 * len(X_seq))
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]
    
    print(f'🔧 Lookback: {LOOKBACK} hours')
    print(f'📊 Training: {len(X_train)} samples | Testing: {len(X_test)} samples')
    
    # Build model
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOKBACK, len(features))),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse')
    model.summary()
    
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    
    print('\n🚀 Training LSTM...')
    history = model.fit(
        X_train, y_train,
        epochs=50, batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )
    print('✅ Training complete!')

## 📉 Step 4: Training & Validation Loss

In [ ]:
if READY:
    fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0e1117')
    ax.set_facecolor('#1a1a2e')
    ax.plot(history.history['loss'], color='#4fc3f7', linewidth=2, label='Training Loss')
    ax.plot(history.history['val_loss'], color='#ef5350', linewidth=2, label='Validation Loss')
    ax.set_xlabel('Epoch', color='#90a4ae', fontsize=12)
    ax.set_ylabel('MSE Loss', color='#90a4ae', fontsize=12)
    ax.set_title('📉 LSTM Training & Validation Loss', color='white', fontsize=14, fontweight='bold')
    ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=11)
    ax.tick_params(colors='#90a4ae')
    ax.grid(True, alpha=0.1, color='#90a4ae')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')
    plt.tight_layout()
    plt.savefig('lstm_loss_curve.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()

## 📊 Step 5: Predicted vs Actual (Test Set)

In [ ]:
if READY:
    y_pred = model.predict(X_test, verbose=0)
    y_pred_inv = scaler_y.inverse_transform(y_pred)
    y_test_inv = scaler_y.inverse_transform(y_test)
    
    r2 = r2_score(y_test_inv, y_pred_inv)
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    
    print(f'\n=== LSTM Forecast Performance ===')
    print(f'  R²   : {r2:.4f}')
    print(f'  RMSE : {rmse:.3f} mm')
    print(f'  MAE  : {mae:.3f} mm')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0e1117')
    
    # Time series
    ax = axes[0]
    ax.set_facecolor('#1a1a2e')
    ax.plot(y_test_inv, color='#4fc3f7', linewidth=1.5, label='Actual', alpha=0.8)
    ax.plot(y_pred_inv, color='#ef5350', linewidth=1.5, label='LSTM Predicted', alpha=0.8)
    ax.set_xlabel('Test Samples', color='#90a4ae', fontsize=12)
    ax.set_ylabel('Water Deficit (mm)', color='#90a4ae', fontsize=12)
    ax.set_title(f'📈 LSTM Forecast vs Actual (R²={r2:.3f})', color='white', fontweight='bold')
    ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
    ax.tick_params(colors='#90a4ae')
    ax.grid(True, alpha=0.1, color='#90a4ae')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')
    
    # Scatter
    ax = axes[1]
    ax.set_facecolor('#1a1a2e')
    ax.scatter(y_test_inv, y_pred_inv, color='#4fc3f7', alpha=0.5, s=30, edgecolor='white', linewidth=0.3)
    lims = [min(y_test_inv.min(), y_pred_inv.min()), max(y_test_inv.max(), y_pred_inv.max())]
    ax.plot(lims, lims, '--', color='#ef5350', linewidth=1.5, label='1:1 Perfect Fit')
    ax.set_xlabel('Actual Water Deficit (mm)', color='#90a4ae', fontsize=12)
    ax.set_ylabel('Predicted Water Deficit (mm)', color='#90a4ae', fontsize=12)
    ax.set_title(f'🎯 Prediction Scatter (RMSE={rmse:.2f}mm)', color='white', fontweight='bold')
    ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
    ax.tick_params(colors='#90a4ae')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')
    
    plt.tight_layout()
    plt.savefig('lstm_predictions.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()

## 🌾 Step 6: Per-Field Forecast Analysis

In [ ]:
if READY:
    fields = df['field_name'].unique()
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#0e1117')
    axes = axes.flatten()
    field_colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    
    for i, (field, color) in enumerate(zip(fields[:4], field_colors)):
        ax = axes[i]
        ax.set_facecolor('#1a1a2e')
        
        fdata = df[df['field_name'] == field].groupby('hour')[features + [target]].mean().dropna()
        if len(fdata) < LOOKBACK + 5:
            ax.text(0.5, 0.5, f'Insufficient data\nfor {field}', ha='center', va='center',
                    color='white', transform=ax.transAxes, fontsize=12)
            continue
        
        X_f = scaler_X.transform(fdata[features].values)
        y_f = fdata[target].values
        X_fseq, y_fseq = create_sequences(X_f, scaler_y.transform(fdata[[target]].values), LOOKBACK)
        
        if len(X_fseq) > 0:
            y_fpred = scaler_y.inverse_transform(model.predict(X_fseq, verbose=0))
            y_factual = scaler_y.inverse_transform(y_fseq)
            
            ax.plot(y_factual, color='white', linewidth=1.2, label='Actual', alpha=0.7)
            ax.plot(y_fpred, color=color, linewidth=1.5, label='LSTM Forecast', alpha=0.9)
            fr2 = r2_score(y_factual, y_fpred)
            ax.set_title(f'🌾 {field} (R²={fr2:.3f})', color='white', fontweight='bold')
        
        ax.set_ylabel('Water Deficit (mm)', color='#90a4ae', fontsize=10)
        ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
        ax.tick_params(colors='#90a4ae', labelsize=8)
        ax.grid(True, alpha=0.1, color='#90a4ae')
        for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')
    
    plt.suptitle('🌾 Per-Field LSTM Water Deficit Forecast', color='white', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('lstm_per_field.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()

---

> **💡 Note:** This model's accuracy improves significantly as more data accumulates. With 1 week of data, expect R² ≈ 0.6–0.7. With 1 month, expect R² ≈ 0.85+. With 3 months, expect R² ≈ 0.90+.

<div align='center'>

### 📈 AquaVolt-AI — LSTM Forecasting Complete
*The model learns from your growing dataset in real-time*  
**Umer Tanveer** · PhD Candidate · AWKUM Pakistan  
[GitHub](https://github.com/umertanveer25/aquavolt-ai-pk)

</div>